# 1. Setup: Packages and Global Parameters


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv
import glob
import json
import os

# Folder in Google Drive containing the batch JSON files to combine.
BATCH_DIR = "/content/drive/MyDrive/phd/P4/Data"
BATCH_PATTERN = "batch*.json"
OUTPUT_CSV = os.path.join(BATCH_DIR, "aiid_full_incidents_processed.csv")

FIELDNAMES = [
    "incident_id", "title", "description", "date", "date_modified", "year",
    "developer", "deployer", "harmed", "implicated_systems", "report_count",
    "mit_risk_domain", "mit_risk_subdomain", "mit_entity", "mit_intent", "mit_timing",
    "cset_sector", "cset_lives_lost", "cset_injuries", "cset_location_country",
    "cset_ai_system_description", "cset_entities_raw",
    "gmf_known_ai_goal", "gmf_known_technology", "gmf_known_technical_failure",
    "n_cset_annotators",
]


def names_join(items):
    return "; ".join(x.get("name", "") for x in (items or []) if x)


def find_classification(classifications, namespace, require_publish=False):
    for c in classifications or []:
        if c.get("namespace") == namespace and (not require_publish or c.get("publish")):
            return c
    return None


def count_namespace_prefix(classifications, prefix):
    return sum(1 for c in classifications or [] if str(c.get("namespace", "")).startswith(prefix))


def classification_attr(classification, short_name):
    """Look up an attribute by short_name and decode its JSON-encoded value_json."""
    if not classification:
        return None
    for a in classification.get("attributes", []):
        if a.get("short_name") == short_name:
            try:
                return json.loads(a["value_json"])
            except (TypeError, ValueError):
                return a["value_json"]
    return None


def build_row(incident):
    classifications = incident.get("classifications") or []
    mit = find_classification(classifications, "MIT")
    csetv1 = find_classification(classifications, "CSETv1")
    # GMF ("known AI goal/technology/failure") is only trustworthy once published.
    gmf = find_classification(classifications, "GMF", require_publish=True)

    date = incident.get("date") or ""
    sector = classification_attr(csetv1, "Sector of Deployment")
    lives_lost = classification_attr(csetv1, "Lives Lost")
    injuries = classification_attr(csetv1, "Injuries")

    return {
        "incident_id": incident.get("incident_id"),
        "title": incident.get("title"),
        "description": incident.get("description"),
        "date": date,
        "date_modified": incident.get("date_modified") or "",
        "year": date[:4] if date else "",
        "developer": names_join(incident.get("AllegedDeveloperOfAISystem")),
        "deployer": names_join(incident.get("AllegedDeployerOfAISystem")),
        "harmed": names_join(incident.get("AllegedHarmedOrNearlyHarmedParties")),
        "implicated_systems": names_join(incident.get("implicated_systems")),
        "report_count": len(incident.get("reports") or []),
        "mit_risk_domain": classification_attr(mit, "Risk Domain") or "",
        "mit_risk_subdomain": classification_attr(mit, "Risk Subdomain") or "",
        "mit_entity": classification_attr(mit, "Entity") or "",
        "mit_intent": classification_attr(mit, "Intent") or "",
        "mit_timing": classification_attr(mit, "Timing") or "",
        "cset_sector": "" if sector is None else str(sector),
        "cset_lives_lost": float(lives_lost) if isinstance(lives_lost, (int, float)) else "",
        "cset_injuries": float(injuries) if isinstance(injuries, (int, float)) else "",
        "cset_location_country": classification_attr(csetv1, "Location Country (two letters)") or "",
        "cset_ai_system_description": classification_attr(csetv1, "AI System Description") or "",
        # csetv1 present but attribute missing/null -> "null"; csetv1 absent entirely -> "".
        "cset_entities_raw": json.dumps(classification_attr(csetv1, "Entities")) if csetv1 is not None else "",
        "gmf_known_ai_goal": json.dumps(classification_attr(gmf, "Known AI Goal")) if gmf is not None else "",
        "gmf_known_technology": json.dumps(classification_attr(gmf, "Known AI Technology")) if gmf is not None else "",
        "gmf_known_technical_failure": json.dumps(classification_attr(gmf, "Known AI Technical Failure")) if gmf is not None else "",
        # Count of CSETv1_Annotator-* classification blocks, published or not.
        "n_cset_annotators": count_namespace_prefix(classifications, "CSETv1_Annotator-"),
    }


batch_paths = sorted(glob.glob(os.path.join(BATCH_DIR, BATCH_PATTERN)))
print(f"Found {len(batch_paths)} batch file(s)")

incidents_by_id = {}

for path in batch_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each batch file is a raw GraphQL response: {"data": {"incidents": [...]}}.
    records = next(iter(data["data"].values())) if isinstance(data, dict) and "data" in data else data

    for incident in records:
        incidents_by_id[incident["incident_id"]] = incident

    print(f"  {os.path.basename(path)}: {len(records)} record(s)")

print(f"Total unique incidents: {len(incidents_by_id)}")

rows = [build_row(inc) for inc in incidents_by_id.values()]
rows.sort(key=lambda r: r["incident_id"])

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")

# 2. Descriptive Analysis


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/phd/P4/Data/aiid_full_incidents_processed.csv"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Columns and dtypes:")
print(df.dtypes)

In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
# Summary statistics for numeric columns
df.describe(include="number").T

In [ ]:
# Value counts for object/categorical columns with manageable cardinality
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    n_unique = df[col].nunique(dropna=True)
    if 1 < n_unique <= 30:
        print(f"--- {col} ({n_unique} unique values) ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

In [ ]:
# Incidents over time, if a date-like column is present (e.g. "date")
date_cols = [c for c in df.columns if "date" in c.lower()]

if date_cols:
    date_col = date_cols[0]
    dates = pd.to_datetime(df[date_col], errors="coerce")
    print(f"Using date column: {date_col}")
    print(f"Range: {dates.min()} to {dates.max()}")
    print()
    print("Incidents per year:")
    print(dates.dt.year.value_counts().sort_index())
else:
    print("No date-like column found.")

# 3. Module 1 --- Compositional Dynamics (H1, H1a)

Tests whether the composition of realized AI incidents shifted materially
over the primary analysis window (2018-2025), and whether that shift
reflects displacement of established risk-domain categories or the
emergence of new ones alongside a stable base (H1a).


In [ ]:
import re

PRIMARY_WINDOW = (2018, 2025)
BREAK_YEAR = 2022  # pre = up to and including 2021; post = 2022 onward

df["year"] = pd.to_numeric(df["year"], errors="coerce")

primary = df[df["year"].between(*PRIMARY_WINDOW)].copy()
classified = primary[primary["mit_risk_domain"].notna() & (primary["mit_risk_domain"] != "")].copy()

# MIT risk domains are prefixed "N. <name>" (e.g. "4. Malicious Actors and Misuse");
# sort/display by that leading number rather than alphabetically.
def domain_sort_key(domain):
    m = re.match(r"\s*(\d+)", str(domain))
    return int(m.group(1)) if m else 999

domains = sorted(classified["mit_risk_domain"].unique(), key=domain_sort_key)

n_primary = len(primary)
n_classified = len(classified)
print(f"Primary window {PRIMARY_WINDOW[0]}-{PRIMARY_WINDOW[1]}: n = {n_primary}")
print(f"Classified under MIT taxonomy: n = {n_classified} ({n_classified / n_primary:.1%})")
print()
print("Risk domains:")
for d in domains:
    print(" ", d)

In [ ]:
# Table 2: annual risk-domain composition (% of classified incidents per year)
year_domain_counts = (
    classified.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)[domains]
)
year_totals = year_domain_counts.sum(axis=1)
year_domain_pct = year_domain_counts.div(year_totals, axis=0) * 100

table2 = year_domain_pct.round(1).copy()
table2["n"] = year_totals
table2

In [ ]:
# Table 3: absolute incident counts, pre- vs post-BREAK_YEAR
classified["period"] = classified["year"].apply(
    lambda y: f"Pre-{BREAK_YEAR}" if y < BREAK_YEAR else f"Post-{BREAK_YEAR}"
)

period_counts = classified.groupby(["mit_risk_domain", "period"]).size().unstack(fill_value=0)
pre_col, post_col = f"Pre-{BREAK_YEAR}", f"Post-{BREAK_YEAR}"
period_counts = period_counts.reindex(columns=[pre_col, post_col], fill_value=0).reindex(domains)

pre_total, post_total = period_counts[pre_col].sum(), period_counts[post_col].sum()
pre_pct = period_counts[pre_col] / pre_total * 100
post_pct = period_counts[post_col] / post_total * 100

table3 = pd.DataFrame({
    f"{pre_col} (n={pre_total})": period_counts[pre_col].astype(str) + " (" + pre_pct.round(1).astype(str) + "%)",
    f"{post_col} (n={post_total})": period_counts[post_col].astype(str) + " (" + post_pct.round(1).astype(str) + "%)",
    "Change (count)": period_counts[post_col] - period_counts[pre_col],
    "Change (share, pp)": (post_pct - pre_pct).round(1),
}).sort_values("Change (count)", ascending=False)

table3

In [ ]:
%pip install -q ruptures

import numpy as np
import ruptures as rpt

# Structural break test on the Malicious Actors & Misuse share series.
mam_domain = next(d for d in domains if "malicious actors" in d.lower())
mam_share = year_domain_pct[mam_domain].sort_index()
years = mam_share.index.to_numpy()
signal = mam_share.to_numpy()

# PELT selects its own number of breakpoints via a penalty (higher = fewer
# breaks); binary segmentation is asked directly for a single breakpoint.
detectors = [
    ("PELT", rpt.Pelt(model="l2").fit(signal), {"pen": 3}),
    ("Binary segmentation", rpt.Binseg(model="l2").fit(signal), {"n_bkps": 1}),
]

for algo_name, algo, kwargs in detectors:
    # predict() returns breakpoint indices, the last of which is len(signal) (end marker).
    breakpoints = [b for b in algo.predict(**kwargs) if b < len(signal)]
    break_years = [int(years[b]) for b in breakpoints]
    print(f"{algo_name}: break(s) at start of {break_years}" if break_years else f"{algo_name}: no break detected")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Poisson regression of annual incident count on year, risk domain, and
# their interaction. A significant interaction (via LR test against the
# no-interaction model) rejects parallel category-specific trends.
year_domain_long = (
    classified.groupby(["year", "mit_risk_domain"]).size().reset_index(name="count")
)
year_domain_long["mit_risk_domain"] = pd.Categorical(
    year_domain_long["mit_risk_domain"], categories=domains
)

full_model = smf.glm(
    "count ~ C(year) * C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()
reduced_model = smf.glm(
    "count ~ C(year) + C(mit_risk_domain)", data=year_domain_long, family=sm.families.Poisson()
).fit()

lr_stat = 2 * (full_model.llf - reduced_model.llf)
lr_df = full_model.df_model - reduced_model.df_model
lr_pvalue = chi2.sf(lr_stat, lr_df)

print(f"LR test for parallel trends: LR = {lr_stat:.1f}, df = {lr_df:.0f}, p = {lr_pvalue:.3g}")
print()

# Per-category annual growth rate: exp(year coefficient) from a separate
# log-linear Poisson fit of count ~ year within each domain.
print("Estimated annual growth multiplier by risk domain:")
for d in domains:
    sub = year_domain_long[year_domain_long["mit_risk_domain"] == d]
    m = smf.glm("count ~ year", data=sub, family=sm.families.Poisson()).fit()
    growth = np.exp(m.params["year"])
    print(f"  {d}: {growth:.3f} ({(growth - 1):+.1%} per year)")

In [ ]:
from scipy.stats import norm


def mann_kendall(series):
    """Original (non-seasonal) Mann-Kendall trend test.

    Returns (trend, S, p_value) where trend is 'increasing', 'decreasing',
    or 'no trend' at the 0.05 level.
    """
    x = np.asarray(series, dtype=float)
    n = len(x)
    s = sum(np.sign(x[j] - x[i]) for i in range(n - 1) for j in range(i + 1, n))

    # Variance of S (no tie correction needed: annual incident counts are
    # essentially continuous-valued for n >= ~8 non-degenerate years).
    var_s = n * (n - 1) * (2 * n + 5) / 18

    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p_value = 2 * (1 - norm.cdf(abs(z)))
    if p_value < 0.05:
        trend = "increasing" if s > 0 else "decreasing"
    else:
        trend = "no trend"
    return trend, s, p_value


# Trend test on each domain's full annual absolute-count series (2018-2025).
annual_counts = classified.groupby(["year", "mit_risk_domain"]).size().unstack(fill_value=0)
annual_counts = annual_counts.reindex(
    index=range(PRIMARY_WINDOW[0], PRIMARY_WINDOW[1] + 1), columns=domains, fill_value=0
)

print("Mann-Kendall trend test (annual absolute incident count, 2018-2025):")
for d in domains:
    trend, s, p = mann_kendall(annual_counts[d])
    print(f"  {d}: {trend} (S = {s:.0f}, p = {p:.3g})")

**Not implemented here:** the media-attention robustness check (renormalizing
annual incident counts against an index of AI-related news volume) requires
an external news-volume series not available in this dataset. Add it as a
separate cell once that index is sourced.